# CDT Morris CRISPRi: Network Discovery (Stage 1.5)

## Overview
- **Goal**: Construct gene regulatory networks from CDT Attention Maps and compare with Morris paper
- **Model**: CDT-II Stage 1.5 (DNA + RNA, 2,361 genes including GFI1B)
- **Data**: Morris STINGseq TSS + SNP perturbations (cell-level)

## Stage 1.5 Key Change
- **GFI1B is at index 2360 in RNA Self-Attention [2361×2361]**
- GFI1B hub analysis, trans-target validation, and in silico knockdown all directly available
- 569 known trans-targets from Morris et al. can be quantitatively compared

## Contents
1. Setup & Data Loading
2. Model Definition & Loading
3. Gene Regulatory Network Construction
4. Hub Gene Discovery
5. Community Detection
6. GFI1B Trans-Target Network Validation
7. Gradient-Based Causal Analysis
8. Cross-Attention Network (DNA->Gene)
9. In Silico Perturbation Simulator
10. Morris Paper Comparison Summary
11. Save & Export

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
!pip install h5py tqdm scipy seaborn networkx python-louvain -q

import os
import h5py
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm
from datetime import datetime
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from dataclasses import dataclass
from typing import Optional, Dict, List

try:
    import community as community_louvain
    HAS_LOUVAIN = True
except ImportError:
    HAS_LOUVAIN = False
    print("Warning: python-louvain not available, community detection will be skipped")

class NumpyEncoder(json.JSONEncoder):
    """JSON encoder that handles numpy types."""
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, np.bool_):
            return bool(obj)
        return super().default(obj)

print("Imports done!")

## 2. Paths & Data Loading

In [ ]:
DRIVE_BASE = Path("/content/drive/MyDrive/cdt_data")
MODEL_BASE = Path("/content/drive/MyDrive/cdt_outputs/morris_crispri_stage1_5")
OUTPUT_BASE = Path("/content/drive/MyDrive/cdt_outputs/morris_analysis_stage1_5")
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

TSS_EFFECTS_PATH = DRIVE_BASE / "morris_celllevel_effects_2361.h5"
TSS_ENFORMER_PATH = DRIVE_BASE / "morris_28genes_enformer.h5"
SNP_ENFORMER_PATH = DRIVE_BASE / "morris_snp_enformer.h5"
GENE_LIST_PATH = DRIVE_BASE / "k562_gene_embeddings_aligned.h5"
MODEL_PATH = MODEL_BASE / "cdt_morris_celllevel_best.pt"

# Check for pre-computed attention from NB2
attn_files = sorted(OUTPUT_BASE.glob('attention_results_*.npz'))
ATTN_PATH = attn_files[-1] if attn_files else None

print("Checking files...")
for name, path in [
    ("TSS effects (2361)", TSS_EFFECTS_PATH),
    ("TSS Enformer", TSS_ENFORMER_PATH),
    ("Gene list", GENE_LIST_PATH),
    ("Model (Stage 1.5)", MODEL_PATH),
    ("Attention results (NB2)", ATTN_PATH),
]:
    if path is None:
        print(f"  [NOT FOUND] {name}")
    else:
        status = "OK" if path.exists() else "NOT FOUND"
        print(f"  [{status}] {name}")

In [ ]:
# Load data (Stage 1.5: 2361 genes)
print("Loading CDT gene list...")

# Load base 2360 genes from RNA embeddings, then append GFI1B
with h5py.File(GENE_LIST_PATH, 'r') as f:
    cdt_genes = [g.decode() if isinstance(g, bytes) else g for g in f['gene_names'][:]]
if 'GFI1B' not in cdt_genes:
    cdt_genes.append('GFI1B')
    print("  GFI1B appended to gene list.")
N_GENES = len(cdt_genes)
gene_to_idx = {g: i for i, g in enumerate(cdt_genes)}
assert N_GENES == 2361, f"Expected 2361 genes, got {N_GENES}"
print(f"  CDT genes: {N_GENES}")
print(f"  GFI1B index: {gene_to_idx['GFI1B']}")

print("Loading TSS cell-level effects...")
with h5py.File(TSS_EFFECTS_PATH, 'r') as f:
    tss_log2fc = f['log2fc'][:]
    tss_cell_expr = f['cell_expr'][:]
    tss_target_gene_idx = f['target_gene_idx'][:]
    tss_target_gene_names = [g.decode() if isinstance(g, bytes) else g
                             for g in f['target_gene_names'][:]]
    tss_val_genes = [g.decode() if isinstance(g, bytes) else g
                     for g in f['val_genes'][:]]
    tss_train_genes = [g.decode() if isinstance(g, bytes) else g
                       for g in f['train_genes'][:]]
print(f"  Cells: {tss_log2fc.shape[0]}, Genes: {tss_log2fc.shape[1]}")
print(f"  Val genes: {tss_val_genes}")

print("Loading Enformer embeddings...")
with h5py.File(TSS_ENFORMER_PATH, 'r') as f:
    tss_enformer_emb = f['embeddings'][:]
    tss_enformer_genes = [g.decode() if isinstance(g, bytes) else g
                          for g in f['gene_names'][:]]
tss_gene_to_enformer = {gene: i for i, gene in enumerate(tss_enformer_genes)}
print(f"  TSS Enformer: {tss_enformer_emb.shape}")

print("\nAll data loaded!")

## 3. Model Definition & Loading

In [ ]:
@dataclass
class CDTCRISPRiConfig:
    dna_dim: int = 3072
    dna_seq_len: int = 896
    n_genes: int = 2361  # Stage 1.5: 2360 + GFI1B
    hidden_dim: int = 512
    nhead: int = 8
    dropout: float = 0.3
    dna_self_attn_layers: int = 2
    rna_self_attn_layers: int = 1


class RawExpressionEncoder(nn.Module):
    def __init__(self, n_genes, hidden_dim, dropout=0.1):
        super().__init__()
        self.n_genes = n_genes
        self.hidden_dim = hidden_dim
        self.gene_embedding = nn.Embedding(n_genes, hidden_dim)
        self.expr_projector = nn.Sequential(
            nn.Linear(1, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout)
        )
        self.combine = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim), nn.Dropout(dropout)
        )

    def forward(self, expression):
        batch_size = expression.size(0)
        device = expression.device
        gene_ids = torch.arange(self.n_genes, device=device)
        gene_emb = self.gene_embedding(gene_ids).unsqueeze(0).expand(batch_size, -1, -1)
        expr_emb = self.expr_projector(expression.unsqueeze(-1))
        combined = torch.cat([gene_emb, expr_emb], dim=-1)
        return self.combine(combined)


class SequenceProjector(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.1):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        self.norm = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.norm(self.linear(x)))


class FlashSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, nhead=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.dropout_p = dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        Q = self.q_proj(x).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(Q, K, V, dropout_p=self.dropout_p if self.training else 0.0)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        attn_out = self.out_proj(attn_out)
        x = self.norm1(x + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))
        return x


class FlashCrossAttentionBlock(nn.Module):
    def __init__(self, d_model, nhead=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.dropout_p = dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key_value):
        batch_size, query_len, _ = query.shape
        key_len = key_value.shape[1]
        Q = self.q_proj(query).view(batch_size, query_len, self.nhead, self.head_dim).transpose(1, 2)
        K = self.k_proj(key_value).view(batch_size, key_len, self.nhead, self.head_dim).transpose(1, 2)
        V = self.v_proj(key_value).view(batch_size, key_len, self.nhead, self.head_dim).transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(Q, K, V, dropout_p=self.dropout_p if self.training else 0.0)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, query_len, self.d_model)
        attn_out = self.out_proj(attn_out)
        x = self.norm1(query + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))
        return x


class VirtualCellEmbedderDNARNA(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = 4
        self.head_dim = d_model // self.nhead
        self.dna_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.rna_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.dna_q_proj = nn.Linear(d_model, d_model)
        self.dna_k_proj = nn.Linear(d_model, d_model)
        self.dna_v_proj = nn.Linear(d_model, d_model)
        self.dna_out_proj = nn.Linear(d_model, d_model)
        self.rna_q_proj = nn.Linear(d_model, d_model)
        self.rna_k_proj = nn.Linear(d_model, d_model)
        self.rna_v_proj = nn.Linear(d_model, d_model)
        self.rna_out_proj = nn.Linear(d_model, d_model)
        self.fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model), nn.LayerNorm(d_model)
        )

    def _attention_pool(self, query, key_value, q_proj, k_proj, v_proj, out_proj):
        batch_size = key_value.size(0)
        seq_len = key_value.size(1)
        query = query.expand(batch_size, -1, -1)
        Q = q_proj(query).view(batch_size, 1, self.nhead, self.head_dim).transpose(1, 2)
        K = k_proj(key_value).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        V = v_proj(key_value).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(Q, K, V)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, 1, self.d_model)
        return out_proj(attn_out).squeeze(1)

    def forward(self, dna_encoded, rna_encoded):
        dna_pooled = self._attention_pool(
            self.dna_query, dna_encoded,
            self.dna_q_proj, self.dna_k_proj, self.dna_v_proj, self.dna_out_proj
        )
        rna_pooled = self._attention_pool(
            self.rna_query, rna_encoded,
            self.rna_q_proj, self.rna_k_proj, self.rna_v_proj, self.rna_out_proj
        )
        concat = torch.cat([dna_pooled, rna_pooled], dim=-1)
        return self.fusion(concat)


class CDTCRISPRiModel(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        if config is None:
            config = CDTCRISPRiConfig()
        self.config = config
        self.dna_projector = SequenceProjector(config.dna_dim, config.hidden_dim, config.dropout)
        self.dna_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
            for _ in range(config.dna_self_attn_layers)
        ])
        self.rna_encoder = RawExpressionEncoder(config.n_genes, config.hidden_dim, config.dropout)
        self.rna_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
            for _ in range(config.rna_self_attn_layers)
        ])
        self.dna_to_rna = FlashCrossAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
        self.vce = VirtualCellEmbedderDNARNA(config.hidden_dim, config.dropout)
        self.task_layer = nn.Sequential(
            nn.Linear(config.hidden_dim, config.hidden_dim), nn.GELU(), nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, config.n_genes)
        )

    def forward(self, dna_emb, rna_expr):
        dna = self.dna_projector(dna_emb)
        rna = self.rna_encoder(rna_expr)
        for layer in self.dna_self_attn_layers:
            dna = layer(dna)
        for layer in self.rna_self_attn_layers:
            rna = layer(rna)
        rna = self.dna_to_rna(query=rna, key_value=dna)
        cell_embedding = self.vce(dna, rna)
        effect = self.task_layer(cell_embedding)
        return effect

print("Model defined!")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

config = CDTCRISPRiConfig()
model = CDTCRISPRiModel(config).to(device)
state_dict = torch.load(MODEL_PATH, map_location=device, weights_only=True)
model.load_state_dict(state_dict)
model.eval()

print(f"Model loaded from: {MODEL_PATH}")

## 4. Load or Compute Attention Network

In [ ]:
# Try to load pre-computed attention from NB2
if ATTN_PATH is not None and ATTN_PATH.exists():
    print(f"Loading pre-computed attention from {ATTN_PATH}")
    attn_data = np.load(ATTN_PATH, allow_pickle=True)
    rna_self_attn = attn_data['gfi1b_rna_self_avg']  # [nhead, 2360, 2360]
    print(f"  RNA Self-Attention shape: {rna_self_attn.shape}")
    LOADED_FROM_NB2 = True
else:
    print("No pre-computed attention found. Computing from scratch...")
    LOADED_FROM_NB2 = False
    
    # Re-implement attention extraction for network construction
    # We need RNA self-attention from multiple val genes
    from collections import defaultdict
    
    class SelfAttentionBlockWithWeights(nn.Module):
        def __init__(self, d_model, nhead=8, dropout=0.1):
            super().__init__()
            self.d_model = d_model
            self.nhead = nhead
            self.head_dim = d_model // nhead
            self.q_proj = nn.Linear(d_model, d_model)
            self.k_proj = nn.Linear(d_model, d_model)
            self.v_proj = nn.Linear(d_model, d_model)
            self.out_proj = nn.Linear(d_model, d_model)
            self.ffn = nn.Sequential(
                nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
                nn.Linear(d_model * 4, d_model), nn.Dropout(dropout)
            )
            self.norm1 = nn.LayerNorm(d_model)
            self.norm2 = nn.LayerNorm(d_model)
            self.dropout_layer = nn.Dropout(dropout)

        def forward(self, x, return_attention=False):
            batch_size, seq_len, _ = x.shape
            Q = self.q_proj(x).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
            K = self.k_proj(x).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
            V = self.v_proj(x).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
            if return_attention:
                scale = self.head_dim ** -0.5
                attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * scale
                attn_weights = F.softmax(attn_weights, dim=-1)
                attn_out = torch.matmul(attn_weights, V)
            else:
                attn_out = F.scaled_dot_product_attention(Q, K, V)
                attn_weights = None
            attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
            attn_out = self.out_proj(attn_out)
            x = self.norm1(x + self.dropout_layer(attn_out))
            x = self.norm2(x + self.ffn(x))
            return x, attn_weights

    # Extract RNA self-attention using model internals
    rna_self_attns = []
    for gene in tss_val_genes:
        cells = val_cells_by_gene[gene]
        n_sample = min(5, len(cells))
        dna = tss_enformer_emb[tss_gene_to_enformer[gene]]
        
        for ci in cells[:n_sample]:
            dna_t = torch.from_numpy(dna.astype(np.float32)).unsqueeze(0).to(device)
            rna_t = torch.from_numpy(tss_cell_expr[ci].astype(np.float32)).unsqueeze(0).to(device)
            
            with torch.no_grad():
                dna_enc = model.dna_projector(dna_t)
                rna_enc = model.rna_encoder(rna_t)
                for layer in model.dna_self_attn_layers:
                    dna_enc = layer(dna_enc)
                
                # Manual attention extraction for RNA self-attention
                rna_layer = model.rna_self_attn_layers[0]
                batch_size, seq_len, _ = rna_enc.shape
                Q = rna_layer.q_proj(rna_enc).view(batch_size, seq_len, rna_layer.nhead, rna_layer.head_dim).transpose(1, 2)
                K = rna_layer.k_proj(rna_enc).view(batch_size, seq_len, rna_layer.nhead, rna_layer.head_dim).transpose(1, 2)
                scale = rna_layer.head_dim ** -0.5
                attn = torch.matmul(Q, K.transpose(-2, -1)) * scale
                attn = F.softmax(attn, dim=-1)
                rna_self_attns.append(attn[0].cpu().numpy())
    
    rna_self_attn = np.mean(rna_self_attns, axis=0)  # [nhead, 2360, 2360]
    print(f"  Computed RNA Self-Attention: {rna_self_attn.shape}")

## 5. Gene Regulatory Network Construction

In [ ]:
# Build directed graph from RNA Self-Attention
print("Gene Regulatory Network Construction")
print("=" * 60)

# Average across heads
attn_matrix = rna_self_attn.mean(axis=0)  # [2360, 2360]

# Remove self-loops
np.fill_diagonal(attn_matrix, 0)

# Threshold: top 1%, 5%, 10% edges
for pct_name, pct in [('1%', 0.99), ('5%', 0.95), ('10%', 0.90)]:
    threshold = np.quantile(attn_matrix, pct)
    n_edges = (attn_matrix > threshold).sum()
    print(f"  Top {pct_name}: threshold={threshold:.6f}, edges={n_edges}")

# Use top 5% for main analysis
threshold_5pct = np.quantile(attn_matrix, 0.95)
adj_matrix = (attn_matrix > threshold_5pct).astype(np.float32)
adj_weighted = attn_matrix * adj_matrix

# Build networkx graph
G = nx.DiGraph()
for i in range(N_GENES):
    G.add_node(i, name=cdt_genes[i])

edges = np.argwhere(adj_matrix > 0)
for src, tgt in edges:
    G.add_edge(src, tgt, weight=attn_matrix[src, tgt])

print(f"\nGraph Statistics:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Density: {nx.density(G):.4f}")

## 6. Hub Gene Discovery

In [ ]:
# Hub gene analysis
print("Hub Gene Discovery")
print("=" * 60)

# Out-degree: master regulators (affect many genes)
out_degrees = dict(G.out_degree())
top_out = sorted(out_degrees.items(), key=lambda x: x[1], reverse=True)[:20]

print(f"\nTop 20 Out-Degree Hubs (Master Regulators):")
for rank, (node, deg) in enumerate(top_out):
    print(f"  {rank+1:2d}. {cdt_genes[node]:<12} out-degree={deg}")

# In-degree: highly regulated genes
in_degrees = dict(G.in_degree())
top_in = sorted(in_degrees.items(), key=lambda x: x[1], reverse=True)[:20]

print(f"\nTop 20 In-Degree Hubs (Highly Regulated):")
for rank, (node, deg) in enumerate(top_in):
    print(f"  {rank+1:2d}. {cdt_genes[node]:<12} in-degree={deg}")

# GFI1B hub status
gfi1b_idx = gene_to_idx.get('GFI1B')
if gfi1b_idx is not None:
    print(f"\nGFI1B Hub Status:")
    print(f"  Out-degree: {out_degrees.get(gfi1b_idx, 0)} (rank {sorted(out_degrees.values(), reverse=True).index(out_degrees.get(gfi1b_idx, 0))+1}/{N_GENES})")
    print(f"  In-degree: {in_degrees.get(gfi1b_idx, 0)} (rank {sorted(in_degrees.values(), reverse=True).index(in_degrees.get(gfi1b_idx, 0))+1}/{N_GENES})")

In [ ]:
# Betweenness centrality (on undirected version for efficiency)
print("\nBetweenness Centrality (top 20):")
G_undirected = G.to_undirected()
betweenness = nx.betweenness_centrality(G_undirected, k=min(500, N_GENES))

top_between = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:20]
for rank, (node, bc) in enumerate(top_between):
    print(f"  {rank+1:2d}. {cdt_genes[node]:<12} betweenness={bc:.6f}")

In [ ]:
# Degree distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
out_vals = list(out_degrees.values())
ax.hist(out_vals, bins=50, color='steelblue', alpha=0.7)
ax.set_xlabel('Out-Degree')
ax.set_ylabel('Count')
ax.set_title('Out-Degree Distribution (Master Regulators)')
ax.axvline(np.mean(out_vals), color='red', linestyle='--', label=f'Mean={np.mean(out_vals):.1f}')
ax.legend()

ax = axes[1]
in_vals = list(in_degrees.values())
ax.hist(in_vals, bins=50, color='coral', alpha=0.7)
ax.set_xlabel('In-Degree')
ax.set_ylabel('Count')
ax.set_title('In-Degree Distribution (Regulated Genes)')
ax.axvline(np.mean(in_vals), color='red', linestyle='--', label=f'Mean={np.mean(in_vals):.1f}')
ax.legend()

plt.suptitle('CDT Gene Regulatory Network: Degree Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_BASE / 'network_degree_distribution.png', dpi=150, bbox_inches='tight')
plt.savefig(OUTPUT_BASE / 'network_degree_distribution.pdf', bbox_inches='tight')
plt.show()

## 7. Community Detection

In [ ]:
# Community detection using Louvain
print("Community Detection")
print("=" * 60)

_local_has_louvain = HAS_LOUVAIN

if _local_has_louvain:
    try:
        import community.community_louvain
        _best_partition_func = community.community_louvain.best_partition
    except (ImportError, AttributeError) as e:
        print(f"  Warning: best_partition not found. Error: {e}. Skipping.")
        _local_has_louvain = False

if _local_has_louvain:
    partition = _best_partition_func(G_undirected, resolution=1.0)
    n_communities = len(set(partition.values()))
    print(f"  Communities found: {n_communities}")

    from collections import Counter
    comm_sizes = Counter(partition.values())
    print(f"  Sizes: {dict(comm_sizes.most_common(10))}")

    # GFI1B community
    if gfi1b_idx is not None:
        gfi1b_comm = partition.get(gfi1b_idx)
        comm_genes = [cdt_genes[node] for node, comm in partition.items() if comm == gfi1b_comm]
        print(f"\n  GFI1B community (ID={gfi1b_comm}):")
        print(f"    Size: {len(comm_genes)} genes")
        print(f"    Sample genes: {comm_genes[:20]}")
else:
    print("  Louvain not available, skipping community detection")
    partition = None

In [ ]:
# Visualize GFI1B subnetwork
if gfi1b_idx is not None:
    # Get GFI1B neighbors (1-hop)
    successors = list(G.successors(gfi1b_idx))
    predecessors = list(G.predecessors(gfi1b_idx))
    neighbors = set(successors + predecessors)
    
    # Limit to top 30 by edge weight
    neighbor_weights = []
    for n in neighbors:
        w = max(attn_matrix[gfi1b_idx, n], attn_matrix[n, gfi1b_idx])
        neighbor_weights.append((n, w))
    neighbor_weights.sort(key=lambda x: x[1], reverse=True)
    top_neighbors = [n for n, _ in neighbor_weights[:30]]
    
    subgraph_nodes = [gfi1b_idx] + top_neighbors
    subG = G.subgraph(subgraph_nodes).copy()
    
    # Plot
    fig, ax = plt.subplots(figsize=(14, 14))
    pos = nx.spring_layout(subG, seed=42, k=2)
    
    # Node colors: GFI1B = red, targets = blue, regulators = green
    node_colors = []
    for node in subG.nodes():
        if node == gfi1b_idx:
            node_colors.append('red')
        elif node in successors:
            node_colors.append('steelblue')
        else:
            node_colors.append('lightgreen')
    
    # Edge widths proportional to weight
    edge_weights = [subG[u][v]['weight'] * 500 for u, v in subG.edges()]
    
    nx.draw_networkx(subG, pos, ax=ax,
                     labels={n: cdt_genes[n] for n in subG.nodes()},
                     node_color=node_colors,
                     node_size=500,
                     font_size=8,
                     width=edge_weights,
                     edge_color='gray',
                     alpha=0.8,
                     arrows=True,
                     arrowsize=10)
    
    ax.set_title('GFI1B Gene Regulatory Subnetwork (CDT Attention)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_BASE / 'gfi1b_subnetwork.png', dpi=150, bbox_inches='tight')
    plt.savefig(OUTPUT_BASE / 'gfi1b_subnetwork.pdf', bbox_inches='tight')
    plt.show()
    
    print(f"  GFI1B out-edges (targets): {len(successors)}")
    print(f"  GFI1B in-edges (regulators): {len(predecessors)}")

## 8. GFI1B Trans-Target Network Validation

In [ ]:
# Compare CDT attention network with Morris experimental results
print("GFI1B Trans-Target Validation")
print("=" * 60)

if gfi1b_idx is not None:
    # CDT attention-based targets (out-edges from GFI1B)
    gfi1b_attn_profile = attn_matrix[gfi1b_idx, :]  # [2360]
    gfi1b_attn_profile[gfi1b_idx] = 0  # Remove self
    
    # Experimental targets: genes with significant log2FC upon GFI1B knockdown
    gfi1b_gene_idx_in_target = tss_target_gene_names.index('GFI1B')
    gfi1b_cell_mask = tss_target_gene_idx == gfi1b_gene_idx_in_target
    gfi1b_mean_log2fc = tss_log2fc[gfi1b_cell_mask].mean(axis=0)  # [2360]
    
    # Define experimental targets: top N by |log2FC|
    experimental_effect_abs = np.abs(gfi1b_mean_log2fc)
    
    for n_targets in [50, 100, 200, 500]:
        # Top experimental targets
        exp_top = set(np.argsort(experimental_effect_abs)[-n_targets:])
        exp_top.discard(gfi1b_idx)  # Remove self
        
        # Top attention targets
        attn_top = set(np.argsort(gfi1b_attn_profile)[-n_targets:])
        
        overlap = exp_top & attn_top
        
        # Enrichment test (hypergeometric)
        from scipy.stats import hypergeom
        pval = hypergeom.sf(len(overlap) - 1, N_GENES, len(exp_top), len(attn_top))
        
        print(f"  Top {n_targets}: overlap={len(overlap)}/{n_targets}, "
              f"expected={n_targets*n_targets/N_GENES:.1f}, "
              f"enrichment={len(overlap)/(n_targets*n_targets/N_GENES):.2f}x, "
              f"p={pval:.2e}")
    
    # Correlation between attention and experimental effect
    r, p = pearsonr(gfi1b_attn_profile, experimental_effect_abs)
    print(f"\n  Correlation: attention vs |experimental effect|")
    print(f"    Pearson r = {r:.4f} (p = {p:.2e})")

In [ ]:
# Scatter plot: attention vs experimental effect
if gfi1b_idx is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    ax = axes[0]
    ax.scatter(gfi1b_attn_profile, gfi1b_mean_log2fc, alpha=0.3, s=5, c='steelblue')
    ax.set_xlabel('CDT Attention to Gene', fontsize=12)
    ax.set_ylabel('Experimental log2FC', fontsize=12)
    ax.set_title(f'GFI1B: Attention vs Experimental Effect\nr = {r:.4f}')
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
    
    ax = axes[1]
    ax.scatter(gfi1b_attn_profile, experimental_effect_abs, alpha=0.3, s=5, c='coral')
    ax.set_xlabel('CDT Attention to Gene', fontsize=12)
    ax.set_ylabel('|Experimental log2FC|', fontsize=12)
    ax.set_title('GFI1B: Attention vs |Effect|')
    
    plt.suptitle('GFI1B Trans-Target Validation', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_BASE / 'gfi1b_attn_vs_effect.png', dpi=150, bbox_inches='tight')
    plt.savefig(OUTPUT_BASE / 'gfi1b_attn_vs_effect.pdf', bbox_inches='tight')
    plt.show()

## 9. Gradient-Based Causal Analysis

In [ ]:
# Reconstruct val_cells_by_gene mapping
from collections import defaultdict
val_cells_by_gene = defaultdict(list)
for i, target_gene_idx_val in enumerate(tss_target_gene_idx):
    gene_name_val = tss_target_gene_names[target_gene_idx_val]
    if gene_name_val in tss_val_genes:
        val_cells_by_gene[gene_name_val].append(i)

# Gradient-based analysis: d(output) / d(input RNA)
print("Gradient-Based Causal Analysis")
print("=" * 60)

# Use GFI1B as case study
gene = 'GFI1B'
dna = tss_enformer_emb[tss_gene_to_enformer[gene]]
cell_idx = val_cells_by_gene[gene][0]

dna_t = torch.from_numpy(dna.astype(np.float32)).unsqueeze(0).to(device)
rna_t = torch.from_numpy(tss_cell_expr[cell_idx].astype(np.float32)).unsqueeze(0).to(device)
rna_t.requires_grad_(True)

# Compute Jacobian: d(output_j) / d(input_i) for key output genes
print("Computing gradient-based influence map...")

# For efficiency, compute gradients for top 50 most affected output genes
gfi1b_gene_idx_in_target = tss_target_gene_names.index('GFI1B')
gfi1b_cell_mask = tss_target_gene_idx == gfi1b_gene_idx_in_target
gfi1b_mean_effect = np.abs(tss_log2fc[gfi1b_cell_mask].mean(axis=0))
top_output_genes = np.argsort(gfi1b_mean_effect)[-50:]

gradient_map = np.zeros((len(top_output_genes), N_GENES), dtype=np.float32)

for i, out_idx in enumerate(tqdm(top_output_genes, desc="Computing gradients")):
    rna_t.grad = None
    model.zero_grad()
    pred = model(dna_t, rna_t)
    pred[0, out_idx].backward(retain_graph=True)
    gradient_map[i] = rna_t.grad[0].cpu().numpy()

# Mean absolute gradient per input gene (across output genes)
mean_abs_grad = np.abs(gradient_map).mean(axis=0)  # [2360]

print(f"  Gradient map shape: {gradient_map.shape}")
print(f"  Top 10 genes by mean |gradient|:")
top_grad_genes = np.argsort(mean_abs_grad)[-10:][::-1]
for idx in top_grad_genes:
    print(f"    {cdt_genes[idx]:<12} |grad|={mean_abs_grad[idx]:.6f}")

In [ ]:
# Compare Attention Map (correlative) vs Gradient Map (causal)
if gfi1b_idx is not None:
    attn_profile = gfi1b_attn_profile  # [2360] from attention
    grad_profile = mean_abs_grad  # [2360] from gradients
    
    r_attn_grad, p_attn_grad = pearsonr(attn_profile, grad_profile)
    
    print(f"\nAttention vs Gradient Comparison:")
    print(f"  Pearson r = {r_attn_grad:.4f} (p = {p_attn_grad:.2e})")
    
    # Top attention genes vs top gradient genes overlap
    for k in [50, 100, 200]:
        attn_top_k = set(np.argsort(attn_profile)[-k:])
        grad_top_k = set(np.argsort(grad_profile)[-k:])
        overlap = attn_top_k & grad_top_k
        print(f"  Top {k}: overlap={len(overlap)}/{k} ({len(overlap)/k*100:.1f}%)")
    
    # Scatter plot
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(attn_profile, grad_profile, alpha=0.3, s=5, c='steelblue')
    ax.set_xlabel('Attention Score (Correlative)', fontsize=12)
    ax.set_ylabel('Mean |Gradient| (Causal)', fontsize=12)
    ax.set_title(f'GFI1B: Attention vs Gradient\nr = {r_attn_grad:.4f}', fontsize=14)
    plt.tight_layout()
    plt.savefig(OUTPUT_BASE / 'gfi1b_attn_vs_gradient.png', dpi=150, bbox_inches='tight')
    plt.savefig(OUTPUT_BASE / 'gfi1b_attn_vs_gradient.pdf', bbox_inches='tight')
    plt.show()

## 10. Cross-Attention Network (DNA->Gene)

In [ ]:
# DNA->RNA Cross-Attention: which DNA regions affect which genes
print("Cross-Attention Network (DNA->Gene)")
print("=" * 60)

# Load cross-attention from NB2 or compute
cross_attn_profiles = {}

if LOADED_FROM_NB2 and ATTN_PATH is not None:
    for gene in tss_val_genes:
        key = f'cross_attn_{gene}'
        if key in attn_data:
            cross_attn_profiles[gene] = attn_data[key]  # [nhead, 2360, 896]
            print(f"  Loaded {gene} cross-attention from NB2")

# Compute if not loaded
for gene in tss_val_genes:
    if gene not in cross_attn_profiles:
        cells = val_cells_by_gene[gene]
        dna = tss_enformer_emb[tss_gene_to_enformer[gene]]
        n_sample = min(5, len(cells))
        
        cross_attns = []
        for ci in cells[:n_sample]:
            dna_t = torch.from_numpy(dna.astype(np.float32)).unsqueeze(0).to(device)
            rna_t = torch.from_numpy(tss_cell_expr[ci].astype(np.float32)).unsqueeze(0).to(device)
            
            with torch.no_grad():
                dna_enc = model.dna_projector(dna_t)
                rna_enc = model.rna_encoder(rna_t)
                for layer in model.dna_self_attn_layers:
                    dna_enc = layer(dna_enc)
                for layer in model.rna_self_attn_layers:
                    rna_enc = layer(rna_enc)
                
                # Manual cross-attention extraction
                cross_layer = model.dna_to_rna
                batch_size, query_len, _ = rna_enc.shape
                key_len = dna_enc.shape[1]
                Q = cross_layer.q_proj(rna_enc).view(batch_size, query_len, cross_layer.nhead, cross_layer.head_dim).transpose(1, 2)
                K = cross_layer.k_proj(dna_enc).view(batch_size, key_len, cross_layer.nhead, cross_layer.head_dim).transpose(1, 2)
                scale = cross_layer.head_dim ** -0.5
                attn = torch.matmul(Q, K.transpose(-2, -1)) * scale
                attn = F.softmax(attn, dim=-1)
                cross_attns.append(attn[0].cpu().numpy())
        
        cross_attn_profiles[gene] = np.mean(cross_attns, axis=0)
        print(f"  Computed {gene} cross-attention ({n_sample} cells)")

print("\nTSS Focus Analysis:")
for gene in tss_val_genes:
    cross = cross_attn_profiles[gene]  # [nhead, 2360, 896]
    # Mean over heads and genes
    profile = cross.mean(axis=0).mean(axis=0)  # [896]
    peak = np.argmax(profile)
    # Attention within +/- 50 bins of center (TSS)
    center_attn = profile[398:498].sum() / profile.sum()
    print(f"  {gene}: peak=bin {peak}, center focus={center_attn:.1%}")

In [ ]:
# Heatmap: DNA->Gene cross-attention for GFI1B
gene = 'GFI1B'
if gene in cross_attn_profiles:
    cross = cross_attn_profiles[gene].mean(axis=0)  # [2360, 896]
    
    # Top 50 genes by total attention
    gene_total_attn = cross.sum(axis=1)
    top_50 = np.argsort(gene_total_attn)[-50:]
    top_50 = np.sort(top_50)
    top_names = [cdt_genes[i] for i in top_50]
    
    fig, ax = plt.subplots(figsize=(16, 10))
    im = ax.imshow(cross[top_50, :], aspect='auto', cmap='viridis')
    ax.set_xlabel('DNA Position (896 bins, TSS-centered)', fontsize=12)
    ax.set_ylabel('Gene', fontsize=12)
    ax.set_yticks(range(len(top_names)))
    ax.set_yticklabels(top_names, fontsize=6)
    ax.axvline(448, color='red', linestyle='--', alpha=0.5, label='TSS')
    ax.legend(fontsize=10)
    plt.colorbar(im, label='Attention')
    plt.title(f'DNA->Gene Cross-Attention ({gene} TSS): Top 50 Genes', fontsize=14)
    plt.tight_layout()
    plt.savefig(OUTPUT_BASE / f'{gene.lower()}_cross_attn_heatmap.png', dpi=150, bbox_inches='tight')
    plt.savefig(OUTPUT_BASE / f'{gene.lower()}_cross_attn_heatmap.pdf', bbox_inches='tight')
    plt.show()

## 11. In Silico Perturbation Simulator

In [ ]:
# In Silico Perturbation: knockdown a gene and observe predicted effects
print("In Silico Perturbation Simulator")
print("=" * 60)

# Use mean cell expression as baseline
# Define ntc_mean_expr using available data
ntc_mean_expr = tss_cell_expr.mean(axis=0)  # Mean expression across cells
baseline_rna = ntc_mean_expr.copy()  # [2360] log1p CPM

def in_silico_knockdown(model, dna_emb, baseline_rna, target_gene_idx, device, kd_fraction=0.0):
    """Simulate knockdown of a gene by setting its expression to kd_fraction.
    
    Args:
        model: CDT model
        dna_emb: [896, 3072] DNA embedding
        baseline_rna: [2360] baseline expression
        target_gene_idx: index of gene to knock down
        device: torch device
        kd_fraction: fraction of original expression (0 = complete KD)
    
    Returns:
        baseline_pred: [2360] prediction with normal expression
        kd_pred: [2360] prediction with knockdown
        delta: [2360] kd_pred - baseline_pred
    """
    dna_t = torch.from_numpy(dna_emb.astype(np.float32)).unsqueeze(0).to(device)
    
    # Baseline prediction
    rna_baseline = torch.from_numpy(baseline_rna.astype(np.float32)).unsqueeze(0).to(device)
    with torch.no_grad():
        baseline_pred = model(dna_t, rna_baseline)[0].cpu().numpy()
    
    # Knockdown prediction
    rna_kd = baseline_rna.copy()
    rna_kd[target_gene_idx] = baseline_rna[target_gene_idx] * kd_fraction
    rna_kd_t = torch.from_numpy(rna_kd.astype(np.float32)).unsqueeze(0).to(device)
    with torch.no_grad():
        kd_pred = model(dna_t, rna_kd_t)[0].cpu().numpy()
    
    delta = kd_pred - baseline_pred
    return baseline_pred, kd_pred, delta

print("In silico perturbation function defined.")

In [ ]:
# GFI1B in silico knockdown
print("\nGFI1B In Silico Knockdown:")
gene = 'GFI1B'
gfi1b_idx = gene_to_idx.get(gene)

if gfi1b_idx is not None and gene in tss_gene_to_enformer:
    dna = tss_enformer_emb[tss_gene_to_enformer[gene]]
    baseline_pred, kd_pred, delta = in_silico_knockdown(
        model, dna, baseline_rna, gfi1b_idx, device
    )
    
    print(f"  Baseline mean |pred|: {np.abs(baseline_pred).mean():.6f}")
    print(f"  KD mean |pred|: {np.abs(kd_pred).mean():.6f}")
    print(f"  Delta mean |delta|: {np.abs(delta).mean():.6f}")
    
    # Top affected genes
    top_affected = np.argsort(np.abs(delta))[-20:][::-1]
    print(f"\n  Top 20 genes affected by GFI1B knockdown:")
    for rank, idx in enumerate(top_affected):
        print(f"    {rank+1:2d}. {cdt_genes[idx]:<12} delta={delta[idx]:.6f}")
    
    # Compare with actual experimental effect
    actual_mean = tss_log2fc[gfi1b_cell_mask].mean(axis=0)
    r_insil, p_insil = pearsonr(delta, actual_mean)
    print(f"\n  In silico vs experimental: r = {r_insil:.4f} (p = {p_insil:.2e})")

    # Plot
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(actual_mean, delta, alpha=0.3, s=5, c='steelblue')
    ax.set_xlabel('Actual Experimental log2FC', fontsize=12)
    ax.set_ylabel('In Silico Predicted Delta', fontsize=12)
    ax.set_title(f'GFI1B: In Silico vs Experimental\nr = {r_insil:.4f}', fontsize=14)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
    ax.axvline(0, color='gray', linestyle='--', linewidth=0.5)
    plt.tight_layout()
    plt.savefig(OUTPUT_BASE / 'gfi1b_in_silico_vs_experimental.png', dpi=150, bbox_inches='tight')
    plt.savefig(OUTPUT_BASE / 'gfi1b_in_silico_vs_experimental.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# CD52 in silico knockdown
gene = 'CD52'
cd52_idx = gene_to_idx.get(gene)

if cd52_idx is not None and gene in tss_gene_to_enformer:
    print(f"\nCD52 In Silico Knockdown:")
    dna = tss_enformer_emb[tss_gene_to_enformer[gene]]
    baseline_pred, kd_pred, delta = in_silico_knockdown(
        model, dna, baseline_rna, cd52_idx, device
    )
    
    top_affected = np.argsort(np.abs(delta))[-20:][::-1]
    print(f"  Top 20 genes affected by CD52 knockdown:")
    for rank, idx in enumerate(top_affected):
        print(f"    {rank+1:2d}. {cdt_genes[idx]:<12} delta={delta[idx]:.6f}")
    
    # Compare with experimental
    if 'CD52' in val_cells_by_gene:
        cd52_gene_idx_in_target = tss_target_gene_names.index('CD52')
        cd52_cell_mask = tss_target_gene_idx == cd52_gene_idx_in_target
        cd52_actual = tss_log2fc[cd52_cell_mask].mean(axis=0)
        r_cd52, p_cd52 = pearsonr(delta, cd52_actual)
        print(f"\n  In silico vs experimental: r = {r_cd52:.4f} (p = {p_cd52:.2e})")

## 12. Morris Paper Comparison Summary

In [ ]:
# Comprehensive comparison: CDT discovery vs Morris experimental results
print("=" * 60)
print("MORRIS PAPER COMPARISON SUMMARY")
print("=" * 60)

summary_data = []

for gene in tss_val_genes:
    gene_idx = gene_to_idx.get(gene)
    gene_idx_in_target = tss_target_gene_names.index(gene)
    cell_mask = tss_target_gene_idx == gene_idx_in_target
    actual_effect = tss_log2fc[cell_mask].mean(axis=0)
    
    # CDT attention profile
    attn_profile_gene = attn_matrix[gene_idx, :] if gene_idx is not None else np.zeros(N_GENES)
    
    # Experimental top targets
    exp_abs = np.abs(actual_effect)
    exp_top100 = set(np.argsort(exp_abs)[-100:])
    
    # CDT attention top targets
    attn_top100 = set(np.argsort(attn_profile_gene)[-100:])
    
    overlap = exp_top100 & attn_top100
    
    # In silico delta
    if gene_idx is not None and gene in tss_gene_to_enformer:
        dna = tss_enformer_emb[tss_gene_to_enformer[gene]]
        _, _, delta = in_silico_knockdown(model, dna, baseline_rna, gene_idx, device)
        r_insil, _ = pearsonr(delta, actual_effect)
    else:
        r_insil = None
    
    result = {
        'gene': gene,
        'n_cells': int(cell_mask.sum()),
        'mean_abs_effect': float(exp_abs.mean()),
        'top100_overlap': len(overlap),
        'in_silico_r': float(r_insil) if r_insil is not None else None,
        'in_cdt_list': gene_idx is not None,
    }
    summary_data.append(result)
    
    insil_str = f"{r_insil:.4f}" if r_insil is not None else "N/A"
    print(f"  {gene:<8} cells={result['n_cells']:>5} "
          f"overlap={len(overlap):>3}/100 "
          f"in_silico_r={insil_str}")

print(f"\nMean top100 overlap: {np.mean([r['top100_overlap'] for r in summary_data]):.1f}/100")
insil_rs = [r['in_silico_r'] for r in summary_data if r['in_silico_r'] is not None]
if insil_rs:
    print(f"Mean in silico r: {np.mean(insil_rs):.4f}")

## 13. Save & Export

In [ ]:
# Save all results
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Save network as edge list
nx.write_edgelist(G, OUTPUT_BASE / f'gene_network_{timestamp}.edgelist', data=['weight'])

# Save summary
results = {
    'timestamp': timestamp,
    'network_stats': {
        'nodes': G.number_of_nodes(),
        'edges': G.number_of_edges(),
        'density': float(nx.density(G)),
    },
    'val_gene_analysis': summary_data,
    'hub_genes': {
        'top_out_degree': [(cdt_genes[n], int(d)) for n, d in top_out[:10]],
        'top_in_degree': [(cdt_genes[n], int(d)) for n, d in top_in[:10]],
    },
}

with open(OUTPUT_BASE / f'network_results_{timestamp}.json', 'w') as f:
    json.dump(results, f, indent=2, cls=NumpyEncoder)

# Save gradient map
np.savez_compressed(
    OUTPUT_BASE / f'gradient_map_{timestamp}.npz',
    gradient_map=gradient_map,
    top_output_genes=top_output_genes,
    mean_abs_grad=mean_abs_grad,
    cdt_genes=np.array(cdt_genes),
)

print(f"Results saved to {OUTPUT_BASE}")
print(f"  - gene_network_{timestamp}.edgelist")
print(f"  - network_results_{timestamp}.json")
print(f"  - gradient_map_{timestamp}.npz")
print(f"\nAll network discovery analysis complete!")